# Advanced Problems with Solutions: Fibonacci, Iterators, and Generators

This notebook turns the Fibonacci sequence topic into a practice-heavy set of advanced problems.

The focus is on **lazy evaluation**, **generator design**, **iterator best practices**, **infinite streams**, **generator pipelines**, `yield from`, `.send()`, validation, and testing.

> Fibonacci convention used here: `Fib(0) = 1`, `Fib(1) = 1`, so the sequence begins:
>
> `1, 1, 2, 3, 5, 8, 13, ...`


## Best-practice checklist

Use this checklist while solving the problems:

1. Prefer generators when values can be produced one at a time.
2. Avoid recursive Fibonacci for large `n`; recursion is elegant but can be slow and depth-limited.
3. Validate public function inputs early.
4. Do not materialize huge or infinite sequences with `list(...)`.
5. Use `itertools.islice` when sampling from an infinite generator.
6. Remember: a generator object is **single-use**; a generator function is a **factory** for new generator objects.
7. Keep generator state local unless you intentionally need an object with persistent behavior.
8. Write tests for edge cases: `n = 0`, `n = 1`, invalid inputs, and repeated iteration.


In [1]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from heapq import merge
from itertools import islice
from math import isqrt, sqrt
from timeit import timeit
from typing import Callable, Iterable, Iterator, TypeVar, Any

T = TypeVar("T")


## Warm-up example: why recursion is not enough

### Problem 0
Implement a basic recursive Fibonacci function, then compare it with a non-recursive version.

### Solution


In [2]:
def fib_recursive(n: int) -> int:
    """Return Fib(n), using the 1, 1 Fibonacci convention."""
    if n < 0:
        raise ValueError("n must be >= 0")
    if n <= 1:
        return 1
    return fib_recursive(n - 1) + fib_recursive(n - 2)


def fib_iterative(n: int) -> int:
    """Return Fib(n) in O(n) time and O(1) space."""
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if n < 0:
        raise ValueError("n must be >= 0")

    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


assert [fib_recursive(i) for i in range(7)] == [1, 1, 2, 3, 5, 8, 13]
assert [fib_iterative(i) for i in range(7)] == [1, 1, 2, 3, 5, 8, 13]
assert fib_iterative(100) > 0


## Problem 1 — Build a production-quality finite Fibonacci generator

Write `fib_gen(n)` that lazily yields exactly `n` Fibonacci numbers.

Requirements:

- `fib_gen(0)` yields nothing.
- `fib_gen(1)` yields `[1]`.
- `fib_gen(7)` yields `[1, 1, 2, 3, 5, 8, 13]`.
- Negative `n` raises `ValueError`.
- Non-integer `n` raises `TypeError`.
- The function must be lazy: it should use `yield`, not build a list internally.

### Solution


In [3]:
def fib_gen(n: int, *, first: int = 1, second: int = 1) -> Iterator[int]:
    """
    Lazily yield n Fibonacci-like numbers.

    By default, this uses the convention:
        1, 1, 2, 3, 5, ...

    The optional first/second arguments let us reuse the generator for
    Fibonacci-like sequences.
    """
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if n < 0:
        raise ValueError("n must be >= 0")

    a, b = first, second
    for _ in range(n):
        yield a
        a, b = b, a + b


assert list(fib_gen(0)) == []
assert list(fib_gen(1)) == [1]
assert list(fib_gen(2)) == [1, 1]
assert list(fib_gen(7)) == [1, 1, 2, 3, 5, 8, 13]
assert hasattr(fib_gen(3), "__next__")

try:
    list(fib_gen(-1))
except ValueError:
    pass
else:
    raise AssertionError("fib_gen(-1) should raise ValueError")

try:
    list(fib_gen(3.5))
except TypeError:
    pass
else:
    raise AssertionError("fib_gen(3.5) should raise TypeError")


## Problem 2 — Understand generator exhaustion

Create one generator object and consume it twice. Then fix the issue by using the generator function as a factory.

### Solution


In [4]:
one_shot = fib_gen(5)

first_pass = list(one_shot)
second_pass = list(one_shot)

assert first_pass == [1, 1, 2, 3, 5]
assert second_pass == []  # the generator object is exhausted

# Correct approach: call fib_gen again to create a fresh generator object.
assert list(fib_gen(5)) == [1, 1, 2, 3, 5]
assert list(fib_gen(5)) == [1, 1, 2, 3, 5]

# A factory is useful when multiple passes are needed.
def fib_factory(n: int) -> Callable[[], Iterator[int]]:
    return lambda: fib_gen(n)

make_first_five = fib_factory(5)
assert list(make_first_five()) == [1, 1, 2, 3, 5]
assert list(make_first_five()) == [1, 1, 2, 3, 5]


## Problem 3 — Build an infinite Fibonacci stream safely

Write an infinite generator `infinite_fibonacci()`.

Then use `itertools.islice` to safely take only the first `k` values.

### Solution


In [5]:
def infinite_fibonacci(*, first: int = 1, second: int = 1) -> Iterator[int]:
    """Yield Fibonacci-like values forever."""
    a, b = first, second
    while True:
        yield a
        a, b = b, a + b


def take(iterable: Iterable[T], n: int) -> list[T]:
    """Safely take n values from any iterable, including infinite ones."""
    if n < 0:
        raise ValueError("n must be >= 0")
    return list(islice(iterable, n))


assert take(infinite_fibonacci(), 0) == []
assert take(infinite_fibonacci(), 7) == [1, 1, 2, 3, 5, 8, 13]
assert take(infinite_fibonacci(first=2, second=3), 6) == [2, 3, 5, 8, 13, 21]


## Problem 4 — Design a reusable iterable class

A generator object is single-use. Sometimes we want an object that can be iterated over multiple times.

Build `FibonacciSequence(n)` so this works:

```python
seq = FibonacciSequence(5)
list(seq)  # [1, 1, 2, 3, 5]
list(seq)  # [1, 1, 2, 3, 5] again
```

### Solution


In [6]:
@dataclass(frozen=True)
class FibonacciSequence:
    """Reusable iterable that creates a fresh generator each time __iter__ is called."""

    n: int
    first: int = 1
    second: int = 1

    def __post_init__(self) -> None:
        if not isinstance(self.n, int):
            raise TypeError("n must be an int")
        if self.n < 0:
            raise ValueError("n must be >= 0")

    def __iter__(self) -> Iterator[int]:
        return fib_gen(self.n, first=self.first, second=self.second)

    def __len__(self) -> int:
        return self.n


seq = FibonacciSequence(7)
assert list(seq) == [1, 1, 2, 3, 5, 8, 13]
assert list(seq) == [1, 1, 2, 3, 5, 8, 13]  # reusable
assert len(seq) == 7


## Problem 5 — Compute adjacent ratios lazily

The ratio of consecutive Fibonacci numbers approaches the golden ratio.

Write `adjacent_ratios(iterable)` so it yields:

```python
current / previous
```

for each adjacent pair.

### Solution


In [7]:
def adjacent_ratios(iterable: Iterable[int]) -> Iterator[float]:
    """Yield current / previous for adjacent values in an iterable."""
    iterator = iter(iterable)
    try:
        previous = next(iterator)
    except StopIteration:
        return

    for current in iterator:
        yield current / previous
        previous = current


ratios = list(adjacent_ratios(fib_gen(10)))
phi = (1 + sqrt(5)) / 2

assert ratios[:4] == [1.0, 2.0, 1.5, 5 / 3]
assert abs(ratios[-1] - phi) < 0.02


## Problem 6 — Create a lazy running-total pipeline

Write a reusable generator `running_totals(iterable)`.

Then apply it to Fibonacci numbers.

### Solution


In [8]:
def running_totals(iterable: Iterable[int]) -> Iterator[int]:
    """Yield cumulative totals lazily."""
    total = 0
    for value in iterable:
        total += value
        yield total


assert list(running_totals([])) == []
assert list(running_totals([10, -3, 5])) == [10, 7, 12]
assert list(running_totals(fib_gen(7))) == [1, 2, 4, 7, 12, 20, 33]

# Identity for this 1,1 convention:
# sum(Fib(0)..Fib(n)) == Fib(n + 2) - 1
for n in range(1, 12):
    assert sum(fib_gen(n)) == fib_iterative(n + 1) - 1


## Problem 7 — Filter an infinite stream without hanging

Write a lazy function that returns all even Fibonacci numbers below a limit.

Important: do **not** call `list(infinite_fibonacci())`.

### Solution


In [9]:
def take_while(iterable: Iterable[T], predicate: Callable[[T], bool]) -> Iterator[T]:
    """Yield values while predicate(value) is True, then stop."""
    for value in iterable:
        if not predicate(value):
            return
        yield value


def even_fibonacci_below(limit: int) -> Iterator[int]:
    """Yield even Fibonacci numbers below limit."""
    fibs_under_limit = take_while(infinite_fibonacci(), lambda x: x < limit)
    return (value for value in fibs_under_limit if value % 2 == 0)


assert list(even_fibonacci_below(1)) == []
assert list(even_fibonacci_below(10)) == [2, 8]
assert list(even_fibonacci_below(100)) == [2, 8, 34]
assert sum(even_fibonacci_below(4_000_000)) == 4_613_732


## Problem 8 — Merge multiple sorted Fibonacci-like streams

Use `heapq.merge` to combine multiple sorted iterables lazily.

Then remove duplicates lazily.

### Solution


In [10]:
def unique_sorted(*iterables: Iterable[int]) -> Iterator[int]:
    """Merge sorted iterables and yield unique values lazily."""
    sentinel = object()
    previous: Any = sentinel

    for value in merge(*iterables):
        if previous is sentinel or value != previous:
            yield value
            previous = value


base = fib_gen(7)                         # 1, 1, 2, 3, 5, 8, 13
shifted = fib_gen(5, first=2, second=3)    # 2, 3, 5, 8, 13

assert list(unique_sorted(base, shifted)) == [1, 2, 3, 5, 8, 13]

# More streams, still lazy.
merged_sample = list(unique_sorted(
    fib_gen(8),
    fib_gen(6, first=2, second=3),
    fib_gen(5, first=4, second=7),
))

assert merged_sample == sorted(set(merged_sample))


## Problem 9 — Validate a Fibonacci recurrence from a stream

Write a general `windowed(iterable, size)` generator.

Then use it to detect where a sequence violates the Fibonacci rule:

```python
current == previous + previous_previous
```

### Solution


In [11]:
def windowed(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    """Yield fixed-size sliding windows from an iterable."""
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    window = deque(islice(iterator, size), maxlen=size)

    if len(window) == size:
        yield tuple(window)

    for value in iterator:
        window.append(value)
        yield tuple(window)


def recurrence_violations(sequence: Iterable[int]) -> Iterator[tuple[int, tuple[int, int, int]]]:
    """
    Yield (index, triple) for each violation.

    index is the index of the third value in the triple.
    """
    for index, (a, b, c) in enumerate(windowed(sequence, 3), start=2):
        if a + b != c:
            yield index, (a, b, c)


assert list(windowed([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)]
assert list(recurrence_violations([1, 1, 2, 3, 5, 8, 13])) == []
assert list(recurrence_violations([1, 1, 2, 4, 5])) == [
    (3, (1, 2, 4)),
    (4, (2, 4, 5)),
]


## Problem 10 — Use `yield from` to flatten pages of Fibonacci numbers

Sometimes data arrives in chunks/pages.

Write:

- `fib_pages(total, page_size)` → yields lists of Fibonacci numbers.
- `flatten_pages(pages)` → lazily flattens those pages using `yield from`.

### Solution


In [12]:
def fib_pages(total: int, page_size: int) -> Iterator[list[int]]:
    """Yield Fibonacci numbers in pages of at most page_size values."""
    if total < 0:
        raise ValueError("total must be >= 0")
    if page_size <= 0:
        raise ValueError("page_size must be positive")

    source = fib_gen(total)
    while True:
        page = list(islice(source, page_size))
        if not page:
            return
        yield page


def flatten_pages(pages: Iterable[Iterable[T]]) -> Iterator[T]:
    """Flatten an iterable of iterables lazily."""
    for page in pages:
        yield from page


assert list(fib_pages(0, 3)) == []
assert list(fib_pages(7, 3)) == [[1, 1, 2], [3, 5, 8], [13]]
assert list(flatten_pages(fib_pages(7, 3))) == list(fib_gen(7))


## Problem 11 — Stream rows and keep only Fibonacci values

Build a tiny CSV-like streaming pipeline.

Requirements:

1. Ignore blank lines and comments.
2. Ignore malformed rows.
3. Parse `id,value` rows.
4. Yield only rows whose value is a Fibonacci number.

### Solution


In [13]:
def is_perfect_square(n: int) -> bool:
    if n < 0:
        return False
    root = isqrt(n)
    return root * root == n


def is_fibonacci_number(n: int) -> bool:
    """
    Return True if n is a non-negative Fibonacci number.

    Uses the standard identity:
        n is Fibonacci iff 5*n*n + 4 or 5*n*n - 4 is a perfect square.
    """
    if not isinstance(n, int):
        return False
    if n < 0:
        return False
    return is_perfect_square(5 * n * n + 4) or is_perfect_square(5 * n * n - 4)


def parse_value_rows(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
    """Parse CSV-like lines of the form id,value."""
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = [part.strip() for part in line.split(",")]
        if parts == ["id", "value"]:
            continue
        if len(parts) != 2:
            continue

        row_id, raw_value = parts
        try:
            value = int(raw_value)
        except ValueError:
            continue

        yield row_id, value


def fibonacci_rows(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
    """Yield parsed rows where value is a Fibonacci number."""
    for row_id, value in parse_value_rows(lines):
        if is_fibonacci_number(value):
            yield row_id, value


sample_lines = [
    "id,value",
    "a,1",
    "b,4",
    "c,13",
    "bad row",
    "d,abc",
    "# ignore this",
    "e,21",
    "f,-8",
]

assert list(parse_value_rows(sample_lines)) == [
    ("a", 1),
    ("b", 4),
    ("c", 13),
    ("e", 21),
    ("f", -8),
]
assert list(fibonacci_rows(sample_lines)) == [("a", 1), ("c", 13), ("e", 21)]


## Problem 12 — Build a controllable Fibonacci generator with `.send()`

Advanced generators can receive values using `.send()`.

Build `fib_controller()` that supports:

- `send(None)` → move to the next Fibonacci number.
- `send("reset")` → reset to the original seed.
- `send(("seed", a, b))` → start from new seed values.
- `send(("skip", k))` → skip `k` transitions.

### Solution


In [14]:
def fib_controller(first: int = 1, second: int = 1) -> Iterator[int]:
    """
    Controllable Fibonacci-like generator.

    Protocol:
        next(g) or g.send(None)      -> advance normally
        g.send("reset")             -> reset to original seed
        g.send(("seed", a, b))      -> change current seed
        g.send(("skip", k))         -> skip k transitions
    """
    original_first, original_second = first, second
    a, b = first, second

    while True:
        command = yield a

        if command is None:
            a, b = b, a + b
        elif command == "reset":
            a, b = original_first, original_second
        elif isinstance(command, tuple) and len(command) == 3 and command[0] == "seed":
            _, a, b = command
        elif isinstance(command, tuple) and len(command) == 2 and command[0] == "skip":
            _, k = command
            if not isinstance(k, int) or k < 0:
                raise ValueError("skip amount must be a non-negative int")
            for _ in range(k):
                a, b = b, a + b
        else:
            raise ValueError(f"unknown command: {command!r}")


g = fib_controller()
assert next(g) == 1
assert g.send(None) == 1
assert g.send(None) == 2
assert g.send(None) == 3
assert g.send("reset") == 1
assert g.send(("seed", 2, 3)) == 2
assert g.send(None) == 3
assert g.send(("skip", 3)) == 13


## Problem 13 — Generalize Fibonacci into a recurrence generator

Write `recurrence_gen(seeds, coeffs, n=None)`.

It should support:

- Fibonacci: seeds `[1, 1]`, coefficients `[1, 1]`.
- Tribonacci: seeds `[0, 0, 1]`, coefficients `[1, 1, 1]`.
- Infinite generation when `n is None`.

Coefficient convention: coefficients are ordered from **most recent** value backward.

### Solution


In [15]:
def recurrence_gen(
    seeds: list[int],
    coeffs: list[int],
    n: int | None = None,
) -> Iterator[int]:
    """
    Generate a linear recurrence lazily.

    Example:
        Fibonacci-like: seeds=[1, 1], coeffs=[1, 1]
        next = 1 * most_recent + 1 * second_most_recent
    """
    if not seeds:
        raise ValueError("seeds must not be empty")
    if len(seeds) != len(coeffs):
        raise ValueError("seeds and coeffs must have the same length")
    if n is not None and n < 0:
        raise ValueError("n must be >= 0 or None")

    history = deque(seeds, maxlen=len(seeds))
    produced = 0

    for seed in seeds:
        if n is not None and produced >= n:
            return
        yield seed
        produced += 1

    while n is None or produced < n:
        next_value = sum(c * x for c, x in zip(coeffs, reversed(history)))
        history.append(next_value)
        yield next_value
        produced += 1


assert list(recurrence_gen([1, 1], [1, 1], 7)) == list(fib_gen(7))
assert list(recurrence_gen([0, 0, 1], [1, 1, 1], 8)) == [0, 0, 1, 1, 2, 4, 7, 13]
assert take(recurrence_gen([1, 1], [1, 1], None), 10) == list(fib_gen(10))


## Problem 14 — Create indexed Fibonacci records lazily

Write a pipeline that yields dictionaries for the first `k` even Fibonacci numbers.

Each dictionary should contain:

- `index`
- `value`
- `running_even_sum`

### Solution


In [16]:
def indexed(iterable: Iterable[T], start: int = 0) -> Iterator[tuple[int, T]]:
    """A small explicit version of enumerate, useful in generator pipelines."""
    index = start
    for value in iterable:
        yield index, value
        index += 1


def first_k_even_fibonacci_records(k: int) -> Iterator[dict[str, int]]:
    """Yield records for the first k even Fibonacci numbers."""
    if k < 0:
        raise ValueError("k must be >= 0")
    if k == 0:
        return

    count = 0
    running_sum = 0

    for index, value in indexed(infinite_fibonacci()):
        if value % 2 != 0:
            continue

        running_sum += value
        yield {
            "index": index,
            "value": value,
            "running_even_sum": running_sum,
        }
        count += 1

        if count == k:
            return


assert list(first_k_even_fibonacci_records(0)) == []
assert list(first_k_even_fibonacci_records(5)) == [
    {"index": 2, "value": 2, "running_even_sum": 2},
    {"index": 5, "value": 8, "running_even_sum": 10},
    {"index": 8, "value": 34, "running_even_sum": 44},
    {"index": 11, "value": 144, "running_even_sum": 188},
    {"index": 14, "value": 610, "running_even_sum": 798},
]


## Problem 15 — Benchmark generator design choices

Compare three approaches for producing the first `5_000` Fibonacci numbers:

1. A reusable iterable class.
2. A generator function.
3. A list-building function.

The goal is not to get identical timings on every machine. The goal is to learn how to compare designs safely.

### Solution


In [17]:
def fib_list(n: int) -> list[int]:
    """Eagerly build a list of n Fibonacci numbers."""
    return list(fib_gen(n))


def benchmark_fib_generation(n: int = 5_000, number: int = 5) -> dict[str, float]:
    """Return timing results for a few generation strategies."""
    results = {
        "reusable_iterable_to_list": timeit(
            "list(FibonacciSequence(n))",
            globals={**globals(), "n": n},
            number=number,
        ),
        "generator_function_to_list": timeit(
            "list(fib_gen(n))",
            globals={**globals(), "n": n},
            number=number,
        ),
        "eager_list_function": timeit(
            "fib_list(n)",
            globals={**globals(), "n": n},
            number=number,
        ),
    }
    return results


results = benchmark_fib_generation(n=1_000, number=3)

assert set(results) == {
    "reusable_iterable_to_list",
    "generator_function_to_list",
    "eager_list_function",
}
assert all(value >= 0 for value in results.values())
results


{'reusable_iterable_to_list': 0.0007615997456014156,
 'generator_function_to_list': 0.0006415997631847858,
 'eager_list_function': 0.0006037997081875801}

## Bonus Challenge — Add your own advanced variants

Try these extensions:

1. Generate every `k`-th Fibonacci number lazily.
2. Generate Fibonacci numbers in a numeric range: `low <= value <= high`.
3. Add type hints and docstrings to all functions.
4. Convert the pipeline problems into unit tests using `unittest` or `pytest`.
5. Rewrite one solution as a custom iterator class, then compare readability with the generator version.


In [18]:
def every_kth_fibonacci(k: int) -> Iterator[int]:
    """Yield every k-th Fibonacci value from the infinite Fibonacci stream."""
    if k <= 0:
        raise ValueError("k must be positive")

    for index, value in indexed(infinite_fibonacci()):
        if index % k == 0:
            yield value


def fibonacci_range(low: int, high: int) -> Iterator[int]:
    """Yield Fibonacci numbers value where low <= value <= high."""
    for value in infinite_fibonacci():
        if value > high:
            return
        if value >= low:
            yield value


assert take(every_kth_fibonacci(3), 5) == [1, 3, 13, 55, 233]
assert list(fibonacci_range(10, 100)) == [13, 21, 34, 55, 89]


# Final review

You practiced:

- Recursive vs iterative Fibonacci.
- Finite and infinite generators.
- Generator exhaustion and generator factories.
- Reusable iterables.
- Lazy pipelines.
- Filtering infinite streams safely.
- `yield from`.
- `.send()`.
- Sliding windows.
- Recurrence generalization.
- Benchmarks and correctness tests.

A strong generator solution is usually:

```python
for value in source:
    if useful(value):
        yield transform(value)
```

Small pieces composed lazily are easier to test, reuse, and reason about.
